# 4.2 — Session Linearization + Profile-Based Scoring

This notebook implements a **fundamentally different pipeline** from notebooks 4 and 4.1:

```
Old pipeline:  raw rows (947k) → fill focus_score per row → linearize → ML dataset
New pipeline:  raw rows (947k) → linearize into sessions → score each session → ML dataset
```

## Why this is better

| Concern | Old approach | **This approach** |
|---|---|---|
| What is scored? | A single sensor reading | An entire 30-min session |
| Features used to assign score | Current value only | Mean + min + max + std + range (full session shape) |
| Sensitivity to noise spikes | Ignored unless that row happens to be sampled | `noise_max` directly penalises noisy sessions |
| CO2 build-up over session | Invisible | Captured by `co2_max`, `co2_mean`, `co2_range` |
| Environmental variability | Ignored | Stability score via `*_std` penalises volatile sessions |
| Dataset size | ~947k rows | ~90k sessions |

## Stage 1 — Linearise
Group raw rows by **room** (`source` + `location_id`), then assign each row to a
**fixed 30-minute time window** using `dt.floor('30min')`.  All readings in the
same room that fall within the same window collapse into **one session row** with
seven aggregates per sensor: `mean`, `min`, `max`, `std`, `latest`, `count`, `range`.

## Stage 2 — Score
Assign a `focus_score` (1–5) to every session using eight **session-aware personas**.
Each persona evaluates:
- **Mean comfort** — were average conditions near ideal?
- **Worst-case** — how bad was the worst reading? (`noise_max`, `co2_max`, `temperature_min`)
- **Stability** — was the environment consistent throughout? (penalises high `*_std`)

In [1]:
from pathlib import Path
import sys
import importlib

import pandas as pd
import numpy as np

MAL_DIR = Path.cwd()
if MAL_DIR.name != "MAL":
    MAL_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "MAL")

if str(MAL_DIR) not in sys.path:
    sys.path.insert(0, str(MAL_DIR))

# Force reload so the kernel always uses the latest version of the script,
# even if it was already imported in a previous run.
import scripts.session_linearize_and_score as _mod
importlib.reload(_mod)

from scripts.session_linearize_and_score import (
    SESSION_PROFILES,
    linearize_sessions,
    score_sessions,
    _overwrite_csv,
)

# Sanity-check: confirm the fixed version (dt.floor) is loaded
import inspect
assert "dt.floor" in inspect.getsource(linearize_sessions), (
    "WRONG VERSION LOADED — expected dt.floor window logic. "
    "Restart the kernel and re-run."
)
print("Module loaded OK (dt.floor window logic confirmed)")

PROCESSED_DIR  = MAL_DIR / "data" / "processed"
INPUT_PATH     = PROCESSED_DIR / "final_mock_dataset.csv"
SESSIONS_PATH  = PROCESSED_DIR / "sessions_linearized_30min.csv"
OUTPUT_PATH    = PROCESSED_DIR / "sessions_scored_30min.csv"

SESSION_GAP_MINUTES = 30
RANDOM_STATE        = 42

INPUT_PATH, OUTPUT_PATH

Module loaded OK (dt.floor window logic confirmed)


(WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/processed/final_mock_dataset.csv'),
 WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/processed/sessions_scored_30min.csv'))

## 1 · Load raw data

In [2]:
raw_df = pd.read_csv(INPUT_PATH, low_memory=False)

pd.DataFrame([{
    "raw_rows":         len(raw_df),
    "sources":          raw_df["source"].nunique(),
    "locations":        raw_df["location_id"].nunique(),
    "features":         ["temperature", "humidity", "noise", "co2", "light"],
    "focus_score_pct_missing": f"{raw_df['focus_score'].isna().mean():.0%}",
}])

,raw_rows,sources,locations,features,focus_score_pct_missing
0,947898,7,60,"[temperature, humidity, noise, co2, light]",100%


In [3]:
# Per-source row counts
raw_df["source"].value_counts().rename_axis("source").reset_index(name="rows")

,source,rows
0,keti_1min_resampled,524639
1,room_measurements,108902
2,homecoach_5min_2024,100325
3,homecoach_5min_2025,100190
4,homecoach_5min_2023,77045
5,homecoach_5min_2026,34132
6,room_conditions,2665


## 2 · Stage 1 — Linearise into sessions

Each raw row is assigned to a **fixed 30-minute time window** using `dt.floor('30min')`.
All rows for the same room that fall within the same window are aggregated into one session row.

Each session produces **one row** with:
- `{feature}_mean/min/max/std/latest/count/range` for every sensor
- `n_readings`, `session_duration_minutes`, `session_start`, `session_end`

> **Note:** ~947k raw rows with 1-min and 5-min cadence across 60 rooms → ~90k session rows.

In [4]:
sessions_df = linearize_sessions(
    raw_df,
    session_gap_minutes=SESSION_GAP_MINUTES,
)

print(f"Raw rows:     {len(raw_df):,}")
print(f"Sessions:     {len(sessions_df):,}")
print(f"Compression:  {len(raw_df)/len(sessions_df):.1f}x fewer rows")
print(f"Columns:      {len(sessions_df.columns)}")

assert len(sessions_df) > 1000, f"Expected thousands of sessions, got {len(sessions_df)} — check for stale imports!"

Raw rows:     947,898
Sessions:     90,475
Compression:  10.5x fewer rows
Columns:      42


In [5]:
# Session size & duration summary
pd.DataFrame([{
    "mean_readings_per_session":    sessions_df["n_readings"].mean().round(1),
    "median_readings_per_session":  sessions_df["n_readings"].median(),
    "mean_duration_min":            sessions_df["session_duration_minutes"].mean().round(1),
    "median_duration_min":          sessions_df["session_duration_minutes"].median().round(1),
    "max_duration_min":             sessions_df["session_duration_minutes"].max().round(1),
}])

,mean_readings_per_session,median_readings_per_session,mean_duration_min,median_duration_min,max_duration_min
0,10.5,6.0,25.4,25.0,29.8


In [6]:
# Sessions per source
sessions_df["source"].value_counts().rename_axis("source").reset_index(name="sessions")

,source,sessions
0,room_measurements,18349
1,keti_1min_resampled,17721
2,homecoach_5min_2025,17516
3,homecoach_5min_2024,17463
4,homecoach_5min_2023,13337
5,homecoach_5min_2026,5999
6,room_conditions,90


In [7]:
# Preview the aggregated feature columns
FEAT_SUFFIXES = ["mean", "min", "max", "std", "latest", "count", "range"]
preview_cols  = ["source", "location_id", "n_readings", "session_duration_minutes"]
for feat in ["temperature", "noise", "co2"]:
    preview_cols += [f"{feat}_{s}" for s in FEAT_SUFFIXES]

sessions_df[preview_cols].head(3)

,source,location_id,n_readings,session_duration_minutes,temperature_mean,temperature_min,temperature_max,temperature_std,temperature_latest,temperature_count,...,noise_latest,noise_count,noise_range,co2_mean,co2_min,co2_max,co2_std,co2_latest,co2_count,co2_range
0,homecoach_5min_2023,homecoach_5min_2023_unknown_location,4,15.0,18.200000,17.7,18.9,0.529150,18.9,4,...,7.615773,4,1.159191,0.000439,0.000434,0.000444,0.000005,0.000441,4,0.000011
1,homecoach_5min_2023,homecoach_5min_2023_unknown_location,6,25.0,19.516667,19.1,19.8,0.278687,19.8,6,...,7.280110,6,0.465857,0.000467,0.000443,0.000541,0.000037,0.000541,6,0.000097
2,homecoach_5min_2023,homecoach_5min_2023_unknown_location,6,25.0,19.566667,19.4,19.8,0.163299,19.4,6,...,6.403124,6,1.128340,0.000860,0.000617,0.001063,0.000168,0.001063,6,0.000446


In [8]:
# Save intermediate sessions (no scores yet) — always overwrite
if SESSIONS_PATH.exists():
    SESSIONS_PATH.unlink()
    print(f"[overwrite] removed existing: {SESSIONS_PATH.name}")
sessions_df.to_csv(SESSIONS_PATH, index=False, mode="w")
print(f"Saved sessions (no scores): {SESSIONS_PATH}")
print(f"Rows written: {len(sessions_df):,}")

Saved sessions (no scores): c:\Users\piotr\Desktop\University\Semester4\SEP4\SEP4\MAL\data\processed\sessions_linearized_30min.csv
Rows written: 90,475


## 3 · Session-aware persona profiles

Eight personas are defined. Each evaluates a session using:
- **Mean comfort** — `gaussian(feature_mean, ideal, tolerance)`
- **Worst-case comfort** — `gaussian(feature_max, ...)` for noise/CO2; `min(gaussian(min), gaussian(max))` for temp/humidity
- **Stability** — `exp(-feature_std / tolerance)` — rewards consistent sessions

Final score weights: `(1 - worst_weight - stab_weight) × mean  +  worst_weight × worst  +  stab_weight × stability`

In [9]:
profile_table = pd.DataFrame([
    {
        "name":              p.name,
        "ideal_temp_C":      p.ideal_temperature,
        "ideal_humidity_%":  p.ideal_humidity,
        "ideal_noise_dB":    p.ideal_noise,
        "ideal_co2_ppm":     p.ideal_co2,
        "worst_case_weight": p.worst_case_weight,
        "variability_weight":p.variability_weight,
        "env_sensitivity":   p.env_sensitivity,
        "description":       p.description,
    }
    for p in SESSION_PROFILES
])
profile_table

,name,ideal_temp_C,ideal_humidity_%,ideal_noise_dB,ideal_co2_ppm,worst_case_weight,variability_weight,env_sensitivity,description
0,thermophile_student,23.0,50.0,35.0,700.0,0.35,0.15,0.65,Needs warmth to concentrate. Cold spikes and t...
1,cool_air_engineer,19.5,45.0,45.0,500.0,0.30,0.10,0.55,"Prefers cool, fresh sessions. Notices when CO2..."
2,noise_sensitive_introvert,21.0,50.0,28.0,650.0,0.45,0.20,0.70,Loses focus the moment noise peaks. Worst-case...
3,balanced_professional,21.5,50.0,40.0,650.0,0.25,0.15,0.45,Evaluates sessions holistically. Neither mean ...
4,humidity_sensitive_artist,22.0,55.0,42.0,700.0,0.20,0.30,0.60,Highly sensitive to humidity swings within a s...
5,co2_sleepy_type,21.0,48.0,38.0,450.0,0.40,0.10,0.65,CO2 accumulation over a session is the primary...
6,outdoor_craver,18.0,55.0,35.0,450.0,0.35,0.15,0.70,"Strongly dislikes warm, stuffy sessions. Worst..."
7,robust_multitasker,21.0,50.0,50.0,800.0,0.15,0.10,0.30,Barely affected by any single dimension. Even ...


## 4 · Stage 2 — Score sessions with profiles

For each session:
1. A profile is picked **uniformly at random**.
2. Comfort is computed from all aggregated features (mean, max, std).
3. A `focus_score` 1–5 is sampled from the profile's environment-adjusted distribution.

The output CSV is **always overwritten** (old file deleted before writing).

In [10]:
scored_df, profile_summary = score_sessions(
    sessions_df,
    random_state=RANDOM_STATE,
)

drop_cols = ["session_id", "source", "location_id", "session_start", "session_end", "n_readings"]
export_df = scored_df.drop(columns=drop_cols, errors="ignore")

# Always overwrite
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print(f"[overwrite] removed existing: {OUTPUT_PATH.name}")
export_df.to_csv(OUTPUT_PATH, index=False, mode="w")
print(f"Saved scored sessions: {OUTPUT_PATH}")
print(f"Total sessions: {len(export_df):,}")

profile_summary

Saved scored sessions: c:\Users\piotr\Desktop\University\Semester4\SEP4\SEP4\MAL\data\processed\sessions_scored_30min.csv
Total sessions: 90,475


,profile_name,description,ideal_temperature,ideal_humidity,ideal_noise,ideal_co2,worst_case_weight,variability_weight,env_sensitivity,sessions_scored
0,thermophile_student,Needs warmth to concentrate. Cold spikes and t...,23.0,50.0,35.0,700.0,0.35,0.15,0.65,11339
1,cool_air_engineer,"Prefers cool, fresh sessions. Notices when CO2...",19.5,45.0,45.0,500.0,0.30,0.10,0.55,11295
2,noise_sensitive_introvert,Loses focus the moment noise peaks. Worst-case...,21.0,50.0,28.0,650.0,0.45,0.20,0.70,11319
3,balanced_professional,Evaluates sessions holistically. Neither mean ...,21.5,50.0,40.0,650.0,0.25,0.15,0.45,11194
4,humidity_sensitive_artist,Highly sensitive to humidity swings within a s...,22.0,55.0,42.0,700.0,0.20,0.30,0.60,11310
5,co2_sleepy_type,CO2 accumulation over a session is the primary...,21.0,48.0,38.0,450.0,0.40,0.10,0.65,11434
6,outdoor_craver,"Strongly dislikes warm, stuffy sessions. Worst...",18.0,55.0,35.0,450.0,0.35,0.15,0.70,11293
7,robust_multitasker,Barely affected by any single dimension. Even ...,21.0,50.0,50.0,800.0,0.15,0.10,0.30,11291


## 5 · Verify output

In [11]:
target_check = pd.DataFrame([{
    "output_csv":               str(OUTPUT_PATH),
    "sessions":                 len(scored_df),
    "feature_columns":          len([c for c in scored_df.columns if any(c.endswith(s) for s in ["_mean","_max","_min","_std","_range","_count","_latest"])]),
    "rows_with_focus_score":    int(scored_df["focus_score"].notna().sum()),
    "rows_missing_focus_score": int(scored_df["focus_score"].isna().sum()),
    "all_scored":               bool(scored_df["focus_score"].notna().all()),
    "min_score":                int(scored_df["focus_score"].min()),
    "max_score":                int(scored_df["focus_score"].max()),
}])
target_check

,output_csv,sessions,feature_columns,rows_with_focus_score,rows_missing_focus_score,all_scored,min_score,max_score
0,c:\Users\piotr\Desktop\University\Semester4\SE...,90475,35,90475,0,True,1,5


In [12]:
assert scored_df["focus_score"].notna().all(),        "Some sessions are missing focus_score!"
assert scored_df["focus_score"].between(1, 5).all(),  "focus_score must be 1-5!"
assert len(scored_df) > 1000, f"Too few sessions: {len(scored_df)} — import issue?"
print(f"All {len(scored_df):,} sessions have a valid 1-5 focus_score.")

All 90,475 sessions have a valid 1-5 focus_score.


## 6 · Focus score distribution

In [13]:
dist = (
    scored_df["focus_score"]
    .value_counts()
    .sort_index()
    .rename_axis("focus_score")
    .reset_index(name="sessions")
)
dist["pct"] = (dist["sessions"] / dist["sessions"].sum() * 100).round(2)
dist

,focus_score,sessions,pct
0,1,12947,14.31
1,2,21928,24.24
2,3,25728,28.44
3,4,20155,22.28
4,5,9717,10.74


In [14]:
# Score distribution by source
scored_df.pivot_table(
    index="source",
    columns="focus_score",
    values="session_id",
    aggfunc="count",
    fill_value=0,
)

focus_score,1,2,3,4,5
source,,,,,
homecoach_5min_2023,1977,3222,3785,2950,1403
homecoach_5min_2024,2496,4204,4983,3887,1893
homecoach_5min_2025,2553,4305,4972,3767,1919
homecoach_5min_2026,844,1439,1721,1367,628
keti_1min_resampled,2517,4334,4954,3997,1919
room_conditions,10,25,23,20,12
room_measurements,2550,4399,5290,4167,1943


## 7 · Environmental feature summary by focus score

Confirm that the scoring makes intuitive sense — higher scores should correlate
with sessions closer to ideal conditions.

In [15]:
# Mean of each aggregate feature grouped by focus_score
feature_means = scored_df.groupby("focus_score")[[
    "temperature_mean", "temperature_max", "temperature_std",
    "humidity_mean",
    "noise_mean",       "noise_max",       "noise_range",
    "co2_mean",         "co2_max",
    "session_duration_minutes", "n_readings",
]].mean().round(2)
feature_means

,temperature_mean,temperature_max,temperature_std,humidity_mean,noise_mean,noise_max,noise_range,co2_mean,co2_max,session_duration_minutes,n_readings
focus_score,,,,,,,,,,,
1,22.56,22.62,0.05,2745.09,6.42,6.70,0.54,0.0,0.0,25.39,10.44
2,22.28,22.34,0.05,2739.22,6.42,6.69,0.54,0.0,0.0,25.40,10.53
3,22.13,22.20,0.05,2734.46,6.42,6.69,0.53,0.0,0.0,25.37,10.39
4,22.09,22.15,0.05,2738.99,6.41,6.69,0.53,0.0,0.0,25.41,10.53
5,22.14,22.20,0.05,2729.62,6.42,6.69,0.53,0.0,0.0,25.44,10.52


In [16]:
# Per-profile score distribution
profile_summary[["profile_name", "sessions_scored", "worst_case_weight",
                 "variability_weight", "env_sensitivity"]]

,profile_name,sessions_scored,worst_case_weight,variability_weight,env_sensitivity
0,thermophile_student,11339,0.35,0.15,0.65
1,cool_air_engineer,11295,0.30,0.10,0.55
2,noise_sensitive_introvert,11319,0.45,0.20,0.70
3,balanced_professional,11194,0.25,0.15,0.45
4,humidity_sensitive_artist,11310,0.20,0.30,0.60
5,co2_sleepy_type,11434,0.40,0.10,0.65
6,outdoor_craver,11293,0.35,0.15,0.70
7,robust_multitasker,11291,0.15,0.10,0.30


## 8 · Final column overview

In [17]:
col_info = pd.DataFrame([
    {"column": col, "dtype": str(scored_df[col].dtype), "non_null": int(scored_df[col].notna().sum())}
    for col in scored_df.columns
])
col_info

,column,dtype,non_null
0,session_id,int64,90475
1,source,str,90475
2,location_id,str,90475
3,session_start,str,90475
4,session_end,str,90475
5,n_readings,int64,90475
6,temperature_mean,float64,90475
7,temperature_min,float64,90475
8,temperature_max,float64,90475
9,temperature_std,float64,90475
